# 9주차 (1) — 군집화 실습 «문제본»

> 이 노트북은 핵심 로직을 비워 둔 학생 배포본이다. `TODO(n)` 이 붙은 자리를 채워 넣으면
> 아래 셀들이 그대로 이어서 동작하도록 만들어져 있다. **셀은 위에서부터 순서대로 실행한다.**

## 학습 목표
1. 거리 기반 군집화의 전처리(표준화)와 k-평균의 동작 원리를 이해한다.
2. Lloyd 알고리즘과 k-means++ 초기화를 **NumPy 브로드캐스팅으로 직접 구현**하고, SSE 가 단조 감소함을 실증한다.
3. 엘보·실루엣·BIC 로 군집 개수 k 를 고르는 세 가지 관점을 비교한다.
4. 계층·밀도(DBSCAN/HDBSCAN)·확률(GMM)·그래프(스펙트럴) 기반 알고리즘의 적용 조건을 구분한다.
5. 내부 지표(실루엣·DB·CH), 외부 지표(ARI·AMI), 그리고 **안정성**으로 군집 결과를 검증한다.

## TODO 목록
| 번호 | 절 | 무엇을 구현하는가 |
|---|---|---|
| 1 | 2 | sklearn `KMeans` 를 구성해 군집 라벨 구하기 |
| 2 | 2-2 | k 별 SSE(`inertia_`)와 **평균 실루엣 계수** 계산 |
| 3 | 3 | **k-means++ 초기화** — D(x)² 에 비례하는 확률 표본추출 |
| 4 | 3 | Lloyd **할당 단계** — 브로드캐스팅 거리 제곱 행렬 + 최근접 중심 할당 + SSE 기록 |
| 5 | 3 | Lloyd **갱신 단계** — 군집 평균으로 중심점 갱신 + 수렴 판정 |
| 6 | 4 | 군집 알고리즘 6종의 하이퍼파라미터 구성 (DBSCAN·HDBSCAN·GMM·스펙트럴 포함) |
| 7 | 4-1 | k-거리 그래프로 DBSCAN 의 `eps` 정하기 |
| 8 | 4-2 | GMM 의 **BIC/AIC** 계산 → BIC 로 K 선택 |
| 9 | 5 | 평가 함수 — 실루엣·DB·CH(내부) + **ARI·AMI**(외부) |
| 10 | 5-1 | 부분표본 반복 군집화로 **안정성 ARI** 분포 구하기 |

## 제출물
- 모든 TODO 를 채워 **끝까지 오류 없이 실행된** 이 노트북 (`.ipynb`, 출력 포함)
- 7절 «제출 과제» 의 필수 1·2번에 대한 답 (노트북 맨 끝에 마크다운 셀로 추가)

## 예상 소요시간
약 150분 (TODO 구현 90분 + 과제 60분)

## 자가 점검
- 3절의 `assert np.all(np.diff(sse_hist) <= 1e-9)` 가 통과하면 TODO(4)·(5) 가 올바르다.
- 3절 끝의 «직접 구현 vs sklearn 의 ARI» 가 **1.0** 에 가까우면 TODO(3)~(5) 전체가 맞다.


# 9주차 — 군집화 (Clustering) 실습  ·  2026 개정판

기계학습특론 | 성명: 최승호 | 이메일: jcn99250@naver.com

## 실습 구성
1. **데이터 준비와 시각화** — 실습 데이터(`clustering_data.csv`)가 없으면 자동 생성
2. **scikit-learn k-평균** — `n_init`, `inertia_`, k-means++ 초기화, 엘보·실루엣으로 k 선택
3. **직접 구현** — NumPy 벡터화 Lloyd 알고리즘 + k-means++ 초기화, SSE 단조 감소 실증
4. **다양한 알고리즘 비교** — 계층(Ward) · DBSCAN · HDBSCAN · GMM(EM) · 스펙트럴
5. **평가와 안정성** — 실루엣/DB/CH, ARI/AMI, 부트스트랩 안정성
6. **(선택) 고차원·임베딩 군집화** — PCA 후 군집, 문장 임베딩 기반 텍스트 군집

## 환경
- Python 3.11+, **scikit-learn ≥ 1.4** (`HDBSCAN` 내장, `n_init='auto'` 기본), NumPy 2.x, matplotlib
- Google Colab 에서 그대로 실행 가능 (추가 설치 불필요, 6절만 선택적으로 설치)

In [ ]:
import sys, numpy as np, pandas as pd, matplotlib, matplotlib.pyplot as plt, sklearn
print("python     :", sys.version.split()[0])
print("numpy      :", np.__version__)
print("pandas     :", pd.__version__)
print("sklearn    :", sklearn.__version__)     # 1.4 이상 권장 (HDBSCAN 내장은 1.3+)

# 한글 폰트 (Colab)
try:
    import matplotlib.font_manager as fm
    for name in ["NanumGothic", "Malgun Gothic", "AppleGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]:
        if any(name in f.name for f in fm.fontManager.ttflist):
            plt.rcParams["font.family"] = name; break
except Exception:
    pass
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 0

---
# 1. 데이터 준비와 시각화

기존 실습과 동일하게 `clustering_data.csv`(2차원 특징 벡터)를 사용한다.
파일이 없으면 같은 성격의 데이터를 `make_blobs` 로 생성하므로 그대로 실행하면 된다.

In [ ]:
from sklearn.datasets import make_blobs
from pathlib import Path

fname = "clustering_data.csv"
if Path(fname).exists():
    feature = np.genfromtxt(fname, delimiter=",")      # 기존 실습과 동일한 로드 방식
    y_true = None
    print("파일에서 로드:", fname)
else:
    feature, y_true = make_blobs(n_samples=500, centers=5, cluster_std=0.9,
                                 random_state=11)     # 참조 라벨 y_true 는 5절 평가에만 사용
    np.savetxt(fname, feature, delimiter=",")
    print("파일이 없어 make_blobs 로 생성 후 저장:", fname)

x1, x2 = feature[:, 0], feature[:, 1]
number_data, number_feature = feature.shape
print(f"number of data    : {number_data}")
print(f"number of feature : {number_feature}")
pd.DataFrame(feature, columns=["x1", "x2"]).describe().round(3)

In [ ]:
plt.figure(figsize=(6, 6))
plt.title("input data")
plt.xlabel("x_1"); plt.ylabel("x_2")
plt.plot(x1, x2, "o", c="#13A4B4", alpha=0.7)
plt.show()

### 스케일링 — 군집화의 첫 단계

거리 기반 군집화는 특징의 **척도**에 그대로 종속된다.
단위가 큰 특징이 거리를 지배하지 않도록 표준화(`StandardScaler`)하는 습관을 들인다.
(이 예제 데이터는 이미 비슷한 척도라 결과가 크게 바뀌지 않지만, 실제 데이터에서는 필수)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
Xs = scaler.fit_transform(feature)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(feature[:, 0], feature[:, 1], s=10, c="#9AA3AF"); ax[0].set_title("원 척도")
ax[1].scatter(Xs[:, 0], Xs[:, 1], s=10, c="#13A4B4");           ax[1].set_title("표준화 후 (z-score)")
for a in ax: a.set_xlabel("x_1"); a.set_ylabel("x_2")
plt.tight_layout(); plt.show()

---
# 2. scikit-learn 을 이용한 k-평균 군집화

### 기존 실습과 달라진 점 (scikit-learn 1.4+)
| 기존 | 2026 개정 | 이유 |
|---|---|---|
| `KMeans(n_clusters=5)` | `KMeans(n_clusters=5, n_init=10, random_state=0)` | 실행마다 결과가 달라지지 않도록 시드 고정, 다중 초기화로 국소 최적 회피 |
| `model.fit(); model.predict()` | `model.fit_predict(X)` | 동일한 데이터에 대해 한 번에 |
| — | `model.inertia_`, `model.n_iter_` | SSE 와 반복 횟수 확인 |

`n_init='auto'` (1.4 기본값)은 `init='k-means++'` 이면 1회, `'random'` 이면 10회를 의미한다.
**수업에서는 비교를 위해 명시적으로 `n_init` 을 지정**한다.

In [ ]:
from sklearn.cluster import KMeans

K = 5
# TODO(1): k-평균 모델을 만들고 표준화 데이터 Xs 의 군집 라벨을 구한다.
#   힌트: KMeans 에 n_clusters=K, init="k-means++", n_init=10, random_state=RANDOM_STATE 를 주고 fit_predict 한다.
#   만들 것: model (적합된 KMeans 객체), labels (각 관측치의 군집 번호, shape (n,))
raise NotImplementedError("TODO(1)")

print("SSE (inertia_) :", round(model.inertia_, 3))
print("반복 횟수       :", model.n_iter_)
print("중심점 (표준화 공간):\n", model.cluster_centers_.round(3))
print("\n군집별 관측치 수:")
print(pd.Series(labels).value_counts().sort_index())


In [ ]:
PALETTE = ["#13A4B4", "#E9A23B", "#D9534F", "#3C9D5D", "#7B5EA7", "#0B2545", "#9AA3AF"]

def plot_cluster(X, label, title="cluster", centers=None, ax=None):
    """군집 결과 시각화 (라벨 -1 = 잡음은 회색 x 로 표시)"""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))
    for k in np.unique(label):
        m = label == k
        if k == -1:
            ax.scatter(X[m, 0], X[m, 1], c="#9AA3AF", s=12, marker="x", label="noise")
        else:
            ax.scatter(X[m, 0], X[m, 1], c=PALETTE[k % len(PALETTE)], s=12, label=f"cluster {k}")
    if centers is not None:
        ax.scatter(centers[:, 0], centers[:, 1], marker="X", c="black", s=160,
                   edgecolor="white", zorder=5, label="centroid")
    ax.set_title(title); ax.set_xlabel("x_1"); ax.set_ylabel("x_2")
    return ax

plot_cluster(Xs, labels, f"k-means (k={K})", model.cluster_centers_)
plt.legend(fontsize=8, loc="best"); plt.show()

### 2-1. 초기화의 영향 — 왜 `n_init` 이 필요한가

동일한 데이터·동일한 k 에서 **초기 중심점만 바꿔** SSE 를 비교해 본다.
`init='random', n_init=1` 로 여러 시드를 돌리면 국소 최적에 빠진 해가 나타난다.

In [ ]:
rows = []
for seed in range(20):
    km_r = KMeans(K, init="random",     n_init=1,  random_state=seed).fit(Xs)
    km_p = KMeans(K, init="k-means++",  n_init=1,  random_state=seed).fit(Xs)
    rows.append({"seed": seed, "random(1회)": km_r.inertia_, "k-means++(1회)": km_p.inertia_})
df_init = pd.DataFrame(rows).set_index("seed")
print(df_init.describe().round(2))

best = KMeans(K, init="k-means++", n_init=50, random_state=RANDOM_STATE).fit(Xs)
print("\nn_init=50 의 SSE (사실상 최선):", round(best.inertia_, 3))

df_init.plot(marker="o", figsize=(8, 3.5))
plt.axhline(best.inertia_, color="black", ls="--", lw=1, label="n_init=50")
plt.ylabel("SSE (inertia)"); plt.title("초기화 방법과 시드에 따른 SSE"); plt.legend(); plt.show()

In [ ]:
# 가장 나쁜 해와 가장 좋은 해를 나란히 시각화
worst_seed = int(df_init["random(1회)"].idxmax())
km_worst = KMeans(K, init="random", n_init=1, random_state=worst_seed).fit(Xs)

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
plot_cluster(Xs, km_worst.labels_, f"random 1회 (seed={worst_seed}) · SSE={km_worst.inertia_:.0f}",
             km_worst.cluster_centers_, ax=ax[0])
plot_cluster(Xs, best.labels_, f"k-means++ n_init=50 · SSE={best.inertia_:.0f}",
             best.cluster_centers_, ax=ax[1])
plt.tight_layout(); plt.show()

### 2-2. k 의 선택 — 엘보와 실루엣

- **엘보**: SSE 는 k 가 커지면 단조 감소하므로 *꺾이는 지점*을 본다 (주관적)
- **실루엣**: `s(i) = (b−a)/max(a,b)`. 평균 실루엣이 최대인 k
- 두 지표가 다른 k 를 가리키는 일이 흔하므로 **해석 가능성·안정성**과 함께 판단한다

In [ ]:
from sklearn.metrics import silhouette_score, silhouette_samples

ks = range(2, 11)
sse, sil = [], []
for k in ks:
    # TODO(2): 각 k 에 대해 k-평균을 적합해 SSE(inertia_)와 «평균 실루엣 계수»를 구해 sse, sil 에 담는다.
    #   힌트: KMeans(k, n_init=10, random_state=RANDOM_STATE).fit(Xs) 후 km.inertia_ 와 silhouette_score(Xs, km.labels_)
    raise NotImplementedError("TODO(2)")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(list(ks), sse, "-o", color="#13A4B4"); ax[0].set(xlabel="k", ylabel="SSE (inertia)", title="엘보(Elbow) 방법")
ax[1].plot(list(ks), sil, "-o", color="#D9534F"); ax[1].set(xlabel="k", ylabel="평균 실루엣", title="실루엣(Silhouette) 방법")
best_k = list(ks)[int(np.argmax(sil))]
for a in ax: a.axvline(best_k, color="#E9A23B", ls="--")
plt.tight_layout(); plt.show()
print("실루엣 최대 k =", best_k)


In [ ]:
# 실루엣 플롯: 군집별 품질을 개별적으로 확인
km = KMeans(best_k, n_init=10, random_state=RANDOM_STATE).fit(Xs)
sv = silhouette_samples(Xs, km.labels_)

fig, ax = plt.subplots(figsize=(6, 4.5))
y_lower = 10
for k in range(best_k):
    vals = np.sort(sv[km.labels_ == k])
    y_upper = y_lower + len(vals)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals, color=PALETTE[k % len(PALETTE)], alpha=0.8)
    ax.text(-0.05, y_lower + 0.5 * len(vals), str(k))
    y_lower = y_upper + 10
ax.axvline(sv.mean(), color="red", ls="--", label=f"평균 {sv.mean():.3f}")
ax.set(xlabel="실루엣 계수", ylabel="관측치 (군집별 정렬)", title=f"실루엣 플롯 (k={best_k})")
ax.legend(); plt.show()

---
# 3. k-평균 직접 구현 (NumPy 벡터화)

기존 실습의 이중 `for` 루프를 **브로드캐스팅**으로 대체한다.

```
X[:, None, :] - C[None, :, :]   # (n,1,d) - (1,k,d) -> (n,k,d)
```
한 번의 연산으로 모든 (관측치, 중심점) 쌍의 거리를 계산한다.

추가로 **k-means++ 초기화**와 **SSE 궤적 기록**을 구현하여 이론(단조 감소)을 수치로 확인한다.

In [ ]:
def kmeans_pp_init(X, k, rng):
    """k-means++ 초기화: D(x)^2 에 비례하는 확률로 중심점을 순차 선택

    반환: (k, d) 모양의 초기 중심점 배열
    """
    C = [X[rng.integers(len(X))]]                      # 첫 중심점은 무작위로 하나
    for _ in range(1, k):
        # TODO(3): 각 관측치에서 «가장 가까운 기존 중심점»까지의 거리 제곱 D(x)^2 을 구하고,
        #          그 값에 비례하는 확률로 다음 중심점을 뽑아 C 에 덧붙인다.
        #   힌트: ((X[:, None, :] - np.array(C)[None, :, :]) ** 2).sum(-1).min(axis=1) 이 D(x)^2, 확률은 d2 / d2.sum()
        raise NotImplementedError("TODO(3)")
    return np.array(C)


def kmeans_scratch(X, k, n_iter=100, tol=1e-8, seed=0, init="k-means++"):
    """Lloyd 알고리즘 직접 구현. 중심점 이동 궤적과 SSE 궤적을 함께 반환

    반환: (label (n,), C (k,d), history (반복+1, k, d), sse_hist (반복,))
    """
    rng = np.random.default_rng(seed)
    C = kmeans_pp_init(X, k, rng) if init == "k-means++" else X[rng.choice(len(X), k, replace=False)]

    history, sse_hist = [C.copy()], []
    for it in range(n_iter):
        # TODO(4): [할당 단계] 브로드캐스팅으로 (n, k) 거리 제곱 행렬 D2 를 만들고,
        #          가장 가까운 중심점 번호 label 을 구한 뒤 현재 SSE 를 sse_hist 에 기록한다.
        #   힌트: X[:, None, :] - C[None, :, :] 는 (n,k,d);  SSE = D2[np.arange(len(X)), label].sum()
        raise NotImplementedError("TODO(4)")

        # TODO(5): [갱신 단계] 각 군집의 «평균»으로 새 중심점 C_new 를 만들어 history 에 기록하고,
        #          중심점 이동량이 tol 보다 작으면 반복을 끝낸다. (빈 군집은 기존 중심점을 그대로 유지)
        #   힌트: X[label == j].mean(axis=0);  np.linalg.norm(C_new - C) < tol 이면 C = C_new 후 break
        raise NotImplementedError("TODO(5)")

    label = ((X[:, None, :] - C[None, :, :]) ** 2).sum(-1).argmin(axis=1)
    return label, C, np.array(history), np.array(sse_hist)


def kmeans_scratch_multi(X, k, n_init=10, seed=0, **kw):
    """sklearn 의 n_init 과 동일하게 여러 번 실행해 SSE 가 가장 작은 해를 채택"""
    best = None
    for s in range(n_init):
        out = kmeans_scratch(X, k, seed=seed + s, **kw)
        if best is None or out[3][-1] < best[3][-1]:
            best = out
    return best


label_1, C_1, hist_1, sse_1 = kmeans_scratch(Xs, K, seed=RANDOM_STATE)          # 1회 실행
label_s, C_s, hist, sse_hist = kmeans_scratch_multi(Xs, K, n_init=10, seed=RANDOM_STATE)  # 10회 중 최선

print("직접 구현 (1회)      SSE :", round(sse_1[-1], 3), " | 반복:", len(sse_1))
print("직접 구현 (n_init=10) SSE :", round(sse_hist[-1], 3), " | 반복:", len(sse_hist))
print("sklearn   (n_init=10) SSE :",
      round(KMeans(K, n_init=10, random_state=RANDOM_STATE).fit(Xs).inertia_, 3))
print()
print("→ 1회 실행은 국소 최적에 빠질 수 있다. 직접 구현에서도 n_init 이 필요한 이유.")


In [ ]:
# 이론 (1) 실증: SSE 는 반복마다 증가하지 않는다 (단조 비증가 → 유한 수렴)
assert np.all(np.diff(sse_hist) <= 1e-9) and np.all(np.diff(sse_1) <= 1e-9), "SSE 가 증가했다면 구현 오류!"

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].plot(sse_1, "-o", color="#9AA3AF", label="1회 실행")
ax[0].plot(sse_hist, "-o", color="#13A4B4", label="n_init=10 중 최선")
ax[0].set(xlabel="iteration", ylabel="SSE", title="SSE 단조 감소 (이론 실증)"); ax[0].legend()

plot_cluster(Xs, label_s, "직접 구현 결과 + 중심점 이동 궤적", C_s, ax=ax[1])
for j in range(K):                                   # 중심점 이동 경로 (기존 실습의 plot_centroid)
    ax[1].plot(hist[:, j, 0], hist[:, j, 1], "-", color="black", lw=1, alpha=0.7)
    ax[1].scatter(hist[0, j, 0], hist[0, j, 1], facecolor="none", edgecolor="black", s=80)
plt.tight_layout(); plt.show()

In [ ]:
# sklearn 결과와 라벨 일치 여부 (라벨 번호는 임의이므로 ARI 로 비교)
from sklearn.metrics import adjusted_rand_score
km_ref = KMeans(K, n_init=10, random_state=RANDOM_STATE).fit(Xs)
print("직접 구현 vs sklearn 의 ARI :", round(adjusted_rand_score(label_s, km_ref.labels_), 4),
      "  (1.0 이면 동일한 분할)")

### 3-1. GMM 의 극한이 k-means 임을 수치로 확인 (심화 과제)

모든 성분의 공분산을 `εI` 로 고정하고 `ε → 0` 으로 보내면
책임도 `γ` 가 0/1 로 수렴하여 k-means 의 경성 할당과 같아진다.

In [ ]:
from sklearn.mixture import GaussianMixture

for eps in [1.0, 1e-1, 1e-2, 1e-3, 1e-4]:
    gm = GaussianMixture(K, covariance_type="spherical", reg_covar=eps,
                         means_init=km_ref.cluster_centers_, max_iter=200,
                         random_state=RANDOM_STATE).fit(Xs)
    gamma_max = gm.predict_proba(Xs).max(axis=1).mean()
    ari = adjusted_rand_score(gm.predict(Xs), km_ref.labels_)
    print(f"reg_covar(ε)={eps:<8} 평균 max γ = {gamma_max:.4f}   k-means 와의 ARI = {ari:.4f}")

---
# 4. 다양한 군집화 알고리즘 비교

`k-means` 가 잘 동작하지 않는 데이터(초승달·동심원·비등방)를 포함해
계층 · 밀도 · 확률 · 그래프 기반 알고리즘을 한눈에 비교한다.

In [ ]:
from sklearn.cluster import AgglomerativeClustering, DBSCAN, HDBSCAN, SpectralClustering
from sklearn.datasets import make_moons, make_circles

def build_algos(k):
    """군집 알고리즘 6종을 {이름: 추정기} 딕셔너리로 반환

    키는 반드시 "k-means", "계층(Ward)", "DBSCAN", "HDBSCAN", "GMM", "스펙트럴" 순서로 만든다.
    """
    # TODO(6): 위 6종의 추정기를 각 알고리즘에 맞는 하이퍼파라미터로 구성해 딕셔너리로 돌려준다.
    #   힌트: KMeans(n_init 지정) · AgglomerativeClustering(linkage="ward") · DBSCAN(eps, min_samples)
    #         · HDBSCAN(min_cluster_size) · GaussianMixture(covariance_type="full")
    #         · SpectralClustering(affinity="nearest_neighbors", n_neighbors=...)  — 시드가 있는 것은 random_state 고정
    raise NotImplementedError("TODO(6)")

datasets = {
    "블롭":   (make_blobs(300, centers=3, random_state=8)[0], 3),
    "초승달": (make_moons(300, noise=0.06, random_state=0)[0], 2),
    "동심원": (make_circles(300, factor=0.4, noise=0.05, random_state=0)[0], 2),
}
names = list(build_algos(3).keys())

fig, axs = plt.subplots(len(datasets), len(names), figsize=(3 * len(names), 3 * len(datasets)))
for i, (dname, (D, k)) in enumerate(datasets.items()):
    Ds = StandardScaler().fit_transform(D)
    for j, (aname, algo) in enumerate(build_algos(k).items()):
        lab = algo.fit_predict(Ds)
        ax = axs[i, j]
        for c in np.unique(lab):
            m = lab == c
            ax.scatter(Ds[m, 0], Ds[m, 1], s=6,
                       c=("#9AA3AF" if c == -1 else PALETTE[c % len(PALETTE)]))
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0: ax.set_title(aname, fontsize=11)
        if j == 0: ax.set_ylabel(dname, fontsize=11)
plt.tight_layout(); plt.show()


### 4-1. DBSCAN 의 `eps` 정하기 — k-거리 그래프

`minPts` 번째 최근접 이웃까지의 거리를 정렬해 그린 뒤, **꺾이는 지점**을 `eps` 로 삼는다.

In [ ]:
from sklearn.neighbors import NearestNeighbors

min_pts = 5
Xm = StandardScaler().fit_transform(make_moons(400, noise=0.08, random_state=1)[0])
dist, _ = NearestNeighbors(n_neighbors=min_pts).fit(Xm).kneighbors(Xm)

# TODO(7): minPts 번째 최근접 이웃까지의 거리를 오름차순 정렬한 k-거리 곡선 kdist 를 만들고,
#          곡선이 «꺾이는 지점»에 해당하는 eps 후보를 float 로 정한다.
#   힌트: dist[:, -1] 이 minPts-거리;  꺾이는 지점은 상위 분위수(예: np.quantile(kdist, 0.95))로 근사한다
raise NotImplementedError("TODO(7)")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(kdist, color="#7B5EA7"); ax[0].set(xlabel="정렬된 관측치", ylabel=f"{min_pts}-거리",
                                              title="k-거리 그래프 (꺾이는 지점 = eps 후보)")
ax[0].axhline(eps, color="#E9A23B", ls="--", label=f"eps ≈ {eps:.2f}"); ax[0].legend()

lab = DBSCAN(eps=eps, min_samples=min_pts).fit_predict(Xm)
plot_cluster(Xm, lab, f"DBSCAN (eps={eps:.2f}, minPts={min_pts})", ax=ax[1])
print("잡음으로 분류된 관측치 수:", int((lab == -1).sum()))
plt.tight_layout(); plt.show()


### 4-2. GMM — 소프트 할당과 BIC 로 K 선택

In [ ]:
bics, aics = [], []
Ks = range(1, 11)
for k in Ks:
    # TODO(8): 성분 수가 k 인 가우시안 혼합 모델을 적합해 BIC 와 AIC 를 bics, aics 에 기록한다.
    #   힌트: GaussianMixture(k, covariance_type="full", n_init=3, random_state=RANDOM_STATE).fit(Xs) → gm.bic(Xs), gm.aic(Xs)
    raise NotImplementedError("TODO(8)")

k_bic = list(Ks)[int(np.argmin(bics))]          # BIC 가 최소인 K 를 채택한다
print("BIC 최소 K =", k_bic)

gm = GaussianMixture(k_bic, covariance_type="full", n_init=3, random_state=RANDOM_STATE).fit(Xs)
proba = gm.predict_proba(Xs)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(list(Ks), bics, "-o", label="BIC"); ax[0].plot(list(Ks), aics, "-o", label="AIC")
ax[0].axvline(k_bic, color="#E9A23B", ls="--"); ax[0].set(xlabel="K", title="GMM 모델 선택"); ax[0].legend()

sc = ax[1].scatter(Xs[:, 0], Xs[:, 1], c=proba.max(axis=1), cmap="magma", s=12)
plt.colorbar(sc, ax=ax[1], label="max 책임도 γ")
ax[1].set_title("소프트 할당의 확신도 (경계에서 낮음)")
plt.tight_layout(); plt.show()

print("\n불확실한 관측치 (max γ < 0.7) 수:", int((proba.max(axis=1) < 0.7).sum()))


---
# 5. 평가와 안정성

- **내부 지표**: 실루엣(↑), Davies–Bouldin(↓), Calinski–Harabasz(↑)
- **외부 지표**: 참조 라벨이 있을 때만 — ARI, AMI (우연 보정됨)
- **안정성**: 부분표본을 반복 군집화해 원 결과와의 ARI 분포를 본다

In [ ]:
from sklearn.metrics import (davies_bouldin_score, calinski_harabasz_score,
                             adjusted_mutual_info_score)

def evaluate(X, lab, y_ref=None):
    """군집 결과를 내부·외부 지표로 평가해 dict 로 반환

    반환 키: n_clusters, noise, (군집이 2개 이상이면) silhouette, davies_bouldin,
             calinski_harabasz, (y_ref 가 있으면) ARI, AMI
    """
    m = lab != -1                                        # 잡음 제외 후 내부 지표 계산
    n_cl = len(set(lab[m]))
    out = {"n_clusters": n_cl, "noise": int((~m).sum())}
    # TODO(9): 군집이 2개 이상일 때 내부 지표 3종을, 참조 라벨 y_ref 가 주어졌을 때 외부 지표 2종을 out 에 담는다.
    #   힌트: 내부 지표는 잡음을 뺀 X[m], lab[m] 으로 계산하고(실루엣↑·DB↓·CH↑),
    #         외부 지표는 adjusted_rand_score(ARI) 와 adjusted_mutual_info_score(AMI) 를 쓴다
    raise NotImplementedError("TODO(9)")
    return out

results = {name: evaluate(Xs, algo.fit_predict(Xs), y_true)
           for name, algo in build_algos(K).items()}
pd.DataFrame(results).T.round(3)


### 5-1. 군집 «안정성» 검사

부분표본을 반복 군집화해 원래 결과와의 ARI 분포를 보면, k 가 데이터에 비해 과하거나 부족할 때 값이 흔들린다.
**TODO(10)** — 부분표본 재군집화와 ARI 계산을 구현한다.

In [ ]:
def stability(X, make_model, n_rep=50, frac=0.8, seed=0):
    """부분표본 반복 군집화 → 겹치는 관측치에 대한 ARI 분포 (길이 n_rep 의 ndarray)"""
    rng = np.random.default_rng(seed)
    base = make_model().fit_predict(X)
    scores = []
    for _ in range(n_rep):
        # TODO(10): 전체의 frac 비율만큼을 «비복원 추출»해 다시 군집화하고,
        #           같은 관측치에 대한 원래 결과(base)와의 ARI 를 scores 에 담는다.
        #   힌트: idx = rng.choice(len(X), int(frac * len(X)), replace=False);  adjusted_rand_score(base[idx], lab)
        raise NotImplementedError("TODO(10)")
    return np.array(scores)

for k in [3, 4, 5, 6]:
    sc = stability(Xs, lambda k=k: KMeans(k, n_init=10, random_state=RANDOM_STATE), n_rep=30)
    print(f"k={k}:  안정성 ARI = {sc.mean():.3f} ± {sc.std():.3f}   (0.75 이상이면 안정적)")


In [ ]:
# k 별 안정성 분포를 상자그림으로
data = [stability(Xs, lambda k=k: KMeans(k, n_init=10, random_state=RANDOM_STATE), n_rep=30)
        for k in range(2, 9)]
plt.figure(figsize=(7, 3.5))
plt.boxplot(data, tick_labels=[str(k) for k in range(2, 9)])
plt.axhline(0.75, color="#D9534F", ls="--", label="안정 기준 0.75")
plt.xlabel("k"); plt.ylabel("부분표본 ARI"); plt.title("k 별 군집 안정성"); plt.legend(); plt.show()

---
# 6. (선택) 고차원 데이터와 임베딩 기반 군집화

## 6-1. PCA 후 군집화 — 고차원의 저주 완화

In [ ]:
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits

digits = load_digits()                                   # 64차원 손글씨 숫자 (n=1797, 10 클래스)
Xd = StandardScaler().fit_transform(digits.data)

lab_raw = KMeans(10, n_init=10, random_state=RANDOM_STATE).fit_predict(Xd)
Z = PCA(n_components=20, random_state=RANDOM_STATE).fit_transform(Xd)
lab_pca = KMeans(10, n_init=10, random_state=RANDOM_STATE).fit_predict(Z)

print("원 공간(64차원) 군집 ARI :", round(adjusted_rand_score(digits.target, lab_raw), 3))
print("PCA(20차원) 군집 ARI     :", round(adjusted_rand_score(digits.target, lab_pca), 3))

Z2 = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Xd)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].scatter(Z2[:, 0], Z2[:, 1], c=digits.target, cmap="tab10", s=8); ax[0].set_title("실제 숫자 라벨")
ax[1].scatter(Z2[:, 0], Z2[:, 1], c=lab_pca,       cmap="tab10", s=8); ax[1].set_title("군집 결과 (PCA 20차원에서)")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## 6-2. 문장 임베딩 기반 텍스트 군집화 (Colab 권장)

정규화된 임베딩에 `KMeans` 를 적용하면 코사인 유사도 기준의 **구형 k-means** 와 같아진다.
`sentence-transformers` 설치가 필요하므로 아래 셀은 선택 실행.

In [ ]:
# !pip install -q sentence-transformers umap-learn
try:
    from sentence_transformers import SentenceTransformer
    docs = [
        "흉부 CT에서 폐렴 소견이 관찰됨", "폐렴 의증으로 항생제 투여 시작", "발열과 기침, 폐 침윤 확인",
        "교통사고로 인한 다발성 골절", "낙상으로 대퇴골 골절 발생", "교통사고 후 두부 외상 의심",
        "혈당 조절 불량한 2형 당뇨", "당화혈색소 상승, 인슐린 조정", "당뇨 환자의 식이 교육 시행",
    ]
    emb = SentenceTransformer("intfloat/multilingual-e5-small").encode(docs, normalize_embeddings=True)

    km_txt = KMeans(3, n_init=10, random_state=RANDOM_STATE).fit(emb)
    for c in range(3):
        print(f"[군집 {c}]")
        for i in np.where(km_txt.labels_ == c)[0]:
            print("   ", docs[i])
except ImportError:
    print("sentence-transformers 가 설치되어 있지 않습니다. 위 pip 주석을 해제한 뒤 실행하세요.")

---
# 7. 정리와 과제

## 정리
1. 군집화는 **유사도 정의 → 최적화 → 검증**의 3단계이며 정답이 없다. 스케일링·거리 측도가 알고리즘 선택보다 먼저다.
2. k-means 는 SSE 를 좌표 하강으로 줄이는 Lloyd 알고리즘이며, **SSE 단조 감소**를 3절에서 직접 확인했다.
3. 초기화에 민감하므로 `k-means++` + `n_init ≥ 10` 이 필수다 (2-1절).
4. 데이터 형태에 따라 DBSCAN/HDBSCAN·GMM·스펙트럴이 필요하다 (4절).
5. k 와 알고리즘은 내부 지표 + **안정성** + 해석 가능성으로 함께 결정한다 (5절).

## 제출 과제
1. **(필수)** 4절의 알고리즘 비교 표를 '비등방(anisotropic)' 데이터에 대해 추가로 작성하고, 각 알고리즘의 성공/실패 이유를 3줄 이내로 설명하시오.
2. **(필수)** 5절의 안정성 함수를 GMM 과 HDBSCAN 에 적용해 k-means 와 비교하고, 어떤 알고리즘이 이 데이터에서 더 안정적인지 논하시오.
3. **(심화)** 3-1절을 확장하여 `ε` 를 10⁻¹ ~ 10⁻⁶ 로 바꾸며 평균 책임도 γ 와 ARI 를 그래프로 그리고, GMM → k-means 수렴을 설명하시오.
4. **(심화)** 본인 연구 데이터(또는 공개 데이터)에 합의 군집화(consensus clustering)를 구현하고, 군집별 특징 요약표와 외부 결과와의 연관을 제시하시오.